# 04 – Tiền xử lý & đặc trưng văn bản; tinh chỉnh Naive Bayes, Logistic Regression

**Người thực hiện:** loc (Huỳnh Tấn Lộc)  
**Mục tiêu:** Trả lời: tiền xử lý / n-gram / min_df ảnh hưởng thế nào? Chọn cấu hình NB, LR tốt nhất.

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> loc`.

**Các bước:**
1. Kiểm chứng lại các con số dữ liệu dùng trong báo cáo (tự giải thích ý nghĩa).
2. Thử word_ngram (1,1), (1,2), (1,3) với NB.
3. Thử min_df 1, 2, 5.
4. Quét alpha của NB (ít nhất 5 giá trị).
5. Quét C của Logistic Regression (ít nhất 5 giá trị), có/không class_weight='balanced'.
6. Vẽ đường F1 theo tham số (lưu bằng save_fig) và chọn cấu hình tốt nhất.

In [ ]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


## 1. Kiểm chứng số liệu dữ liệu
Mỗi con số trong báo cáo phải chỉ được ra ô code sinh ra nó. Chạy ô dưới, rồi **tự viết** 1–2 câu giải thích ý nghĩa từng số.

In [ ]:
df = load_data()
print('Số mẫu:', len(df))
print('Số merchant:', df.merchant_id.nunique())
print('Số cụm thực thể:', df.cluster_id.nunique())
print('Giá trị khuyết:', df.isna().sum().sum())
df.category_label.value_counts()

**Giải thích (tự viết):** …

In [ ]:
# Ví dụ làm sạch tiêu đề: so sánh trước / sau
for t in df.title.sample(8, random_state=0):
    print(f'{t!r:70s} -> {clean_title(t)!r}')

**Nhận xét: hàm clean_title giữ lại gì, bỏ đi gì, tại sao? (tự viết)** …

## 2. Thí nghiệm n-gram (Naive Bayes)
Đổi `word_ngram` rồi chạy lại ô. Mỗi lần chạy = 1 dòng nhật ký.

In [ ]:
res, _ = run(member='loc', model='nb', features='word', params={'alpha': 0.1},
             word_ngram=(1, 1), exp_name='ngram_11', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 3. Thí nghiệm min_df

In [ ]:
res, _ = run(member='loc', model='nb', features='word', params={'alpha': 0.1},
             min_df=1, exp_name='mindf_1', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. Quét alpha (Naive Bayes)
Chạy từng giá trị (hoặc vòng lặp nhỏ) – ghi lại nhận xét sau mỗi đợt.

In [ ]:
for a in [1.0, 0.5]:   # lần sau thử: 0.1, 0.05, 0.01 ...
    run(member='loc', model='nb', features='word', params={'alpha': a}, exp_name=f'alpha_{a}', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 5. Quét C (Logistic Regression)

In [ ]:
run(member='loc', model='lr', features='word', params={'C': 1.0}, exp_name='lr_C1', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 6. Tổng hợp & vẽ
Đọc lại nhật ký của mình, vẽ F1 theo tham số.

In [ ]:
log = pd.read_csv('../experiments/loc/experiment_log.csv')
nb = log[log.model == 'nb'].copy()
nb['alpha'] = nb.params.map(lambda s: json.loads(s).get('alpha'))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(nb.alpha, nb.test_f1_macro, 'o')
ax.set(xscale='log', xlabel='alpha', ylabel='Macro-F1 (Test)', title='NB: Macro-F1 theo alpha')
save_fig(fig, 'loc_nb_alpha', notebook='04_Loc_TienXuLy_NB_LR.ipynb', member='loc', cell='Mục 6',
         y_nghia='(tự viết ý nghĩa hình)')

## 7. Kết luận của Lộc (tự viết)
- Cấu hình chọn: …
- Vì sao: …
- Hạn chế: …